# Módulo 2 — Semantic Game Intelligence (versión limpia)

Buscador de videojuegos en lenguaje natural que combina **búsqueda semántica**, **filtros estructurados** y **ranking híbrido**, con interpretación de la intención mediante reglas (sin LLM).

Esta es la versión final y ordenada del módulo: una sola definición de cada función, sin celdas de depuración ni experimentos repetidos. Las métricas y conclusiones finales están en la última sección.

## Decisiones de diseño

- **8.000 juegos**, 22 columnas. La identidad es `game_id`: hay 484 títulos y ninguno aparece una sola vez, así que el título es solo una pista ambigua.
- Se excluyen `revenue_musd` y `hit` para evitar sesgo comercial. `hit` **no** se usa como etiqueta de relevancia.
- Documentos semánticos construidos solo con `title`, `genre`, `is_indie`, `has_multiplayer`, `early_access` y `controller_support`. No se inventan descripciones.
- Percentiles P10–P90: referencias **blandas** para preferencias, no reglas rígidas.
- Condiciones obligatorias y preferencias se separan y se ponderan distinto.
- Preferencia de precio por **banda P10–P25**: por debajo del P10 se aplica 0,90 para no concentrar los resultados en juegos gratuitos.
- El benchmark usa relevancia 0/1/2 con gold completo (sin muestreo).

In [ ]:
import json
import re
import time
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import chromadb
import sentence_transformers
from sentence_transformers import SentenceTransformer

DATASET_PATH = Path("./data/steam_game_hits.csv")
MODELO = "intfloat/multilingual-e5-base"
EMB_PATH = Path("embeddings_e5_base.npy")
IDS_PATH = Path("embedding_game_ids.npy")
CFG_PATH = Path("embedding_config.json")

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("sentence_transformers:", sentence_transformers.__version__)
print("chromadb:", chromadb.__version__)

## 1. Carga y auditoría del dataset

Comprobación mínima: integridad de `game_id`, títulos ambiguos y percentiles que se usarán como referencias blandas.

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Dimensiones:", df.shape)
print("Filas duplicadas:", df.duplicated().sum(),
      "| game_id duplicados:", df["game_id"].duplicated().sum())
print("Géneros:", df["genre"].nunique(),
      "| Títulos únicos:", df["title"].nunique())
print("Títulos que aparecen una sola vez:", (df["title"].value_counts() == 1).sum())
print("Máxima repetición de un título:", df["title"].value_counts().max())
print("Valores faltantes:", int(df.isna().sum().sum()))

VARS_P = ["price_usd", "file_size_gb", "steam_positive_pct",
          "num_reviews", "owners_k", "peak_concurrent_players",
          "metacritic_score", "dlc_count"]
P = {v: {f"P{q}": float(df[v].quantile(q / 100)) for q in (10, 25, 50, 75, 90)}
     for v in VARS_P}
display(pd.DataFrame(P).T.round(2))

## 2. Documentos semánticos y alineación

`base` se reordena para que fila, documento y embedding compartan el mismo índice. Esa alineación es crítica: el antiguo notebook tenía un error de desplazamiento que se corrige acá de forma explícita.

También se cuantifica el **colapso semántico**: sin título solo existen 192 textos distintos (12 géneros × 2⁴ combinaciones booleanas).

In [ ]:
def build_semantic_document(row):
    genre_text = f"Es un videojuego del género {row['genre']}."
    indie_text = (
        "Es un videojuego desarrollado de forma independiente."
        if row["is_indie"] == 1 else "No es un videojuego independiente."
    )
    multiplayer_text = (
        "Cuenta con modo multijugador." if row["has_multiplayer"] == 1
        else "No cuenta con modo multijugador."
    )
    early_access_text = (
        "Está disponible en acceso anticipado." if row["early_access"] == 1
        else "No está en acceso anticipado."
    )
    controller_text = (
        "Ofrece compatibilidad con controles." if row["controller_support"] == 1
        else "No ofrece compatibilidad con controles."
    )
    return " ".join([
        f"{row['title']}.", genre_text, indie_text,
        multiplayer_text, early_access_text, controller_text,
    ])

semantic_df = df.copy()
semantic_df["semantic_text"] = semantic_df.apply(build_semantic_document, axis=1)
documentos = semantic_df["semantic_text"].tolist()

# Alineación explícita: base conserva el orden de documentos y embeddings
base = df.set_index("game_id").loc[semantic_df["game_id"]].reset_index()
assert np.array_equal(base["game_id"].to_numpy(), semantic_df["game_id"].to_numpy())
id_a_pos = {gid: i for i, gid in enumerate(base["game_id"])}

docs_sin_titulo = [
    re.sub(r"^[^.]*?\.\s*Es un videojuego", "Es un videojuego", d, count=1)
    for d in documentos
]

print("Documentos:", len(documentos), "| únicos:", len(set(documentos)))
print("Documentos únicos sin título:", len(set(docs_sin_titulo)))
print("Combinaciones teóricas:", base["genre"].nunique() * 2**4)
print("Ejemplo:", documentos[0])

## 3. Embeddings

Modelo `intfloat/multilingual-e5-base` (768 dimensiones), normalizados, con prefijos `passage:` para documentos y `query:` para consultas. Si los archivos ya existen se reutilizan; si no, se generan y se guarda su procedencia en `embedding_config.json`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
modelo = SentenceTransformer(MODELO, device=device)
modelo.max_seq_length = 256

if EMB_PATH.exists() and IDS_PATH.exists():
    embeddings = np.load(EMB_PATH)
    ids_guardados = np.load(IDS_PATH, allow_pickle=True).astype(str)
    assert embeddings.shape == (len(base), 768)
    assert np.array_equal(ids_guardados, base["game_id"].to_numpy().astype(str))
    print("Embeddings cargados desde disco:", embeddings.shape)
else:
    t0 = time.time()
    embeddings = modelo.encode(
        ["passage: " + d for d in documentos],
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(EMB_PATH, embeddings)
    np.save(IDS_PATH, base["game_id"].to_numpy())
    print(f"Embeddings generados en {time.time() - t0:.1f} s:", embeddings.shape)

config = {
    "modelo": MODELO,
    "prefijo": "passage: ",
    "max_seq_length": modelo.max_seq_length,
    "normalizado": True,
    "dimension": int(embeddings.shape[1]),
    "filas": int(embeddings.shape[0]),
    "columna_documentos": "semantic_text",
    "dataframe_documentos": "semantic_df",
    "ids_desde": "semantic_df",
}
with open(CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Norma promedio:", round(float(np.linalg.norm(embeddings, axis=1).mean()), 4))

## 4. ChromaDB

Colección persistente `videojuegos_e5` con distancia coseno. Los IDs son strings `G00001`, `G00002`, ... (nunca se convierten con `int()`).

In [ ]:
cliente = chromadb.PersistentClient(path="./chroma_db")
try:
    cliente.delete_collection("videojuegos_e5")
except Exception:
    pass

CAMPOS_METADATA = [
    "title", "genre", "release_year", "price_usd", "file_size_gb",
    "is_indie", "has_multiplayer", "early_access", "controller_support",
    "metacritic_score", "steam_positive_pct", "num_reviews",
    "owners_k", "peak_concurrent_players", "languages_supported",
]

def a_metadata(fila):
    m = {}
    for c in CAMPOS_METADATA:
        v = fila[c]
        if isinstance(v, (np.bool_, bool)):
            v = bool(v)
        elif isinstance(v, np.integer):
            v = int(v)
        elif isinstance(v, np.floating):
            v = float(v)
        elif pd.isna(v):
            v = ""
        elif not isinstance(v, (str, int, float, bool)):
            v = str(v)
        m[c] = v
    return m

coleccion = cliente.get_or_create_collection(
    name="videojuegos_e5",
    metadata={"hnsw:space": "cosine"},
)

BATCH = 1000
for inicio in range(0, len(base), BATCH):
    fin = min(inicio + BATCH, len(base))
    bloque = base.iloc[inicio:fin]
    coleccion.add(
        ids=[str(x) for x in bloque["game_id"]],
        embeddings=embeddings[inicio:fin].tolist(),
        documents=documentos[inicio:fin],
        metadatas=[a_metadata(fila) for _, fila in bloque.iterrows()],
    )

print("Registros en la colección:", coleccion.count())
print("Primer ID:", coleccion.get(limit=1)["ids"])

## 5. Intérprete de consultas (reglas, sin LLM)

### 5.1 Utilidades y reglas

`normalizar`, `resolver_titulo` y `COLUMNAS_CANDIDATO` estaban implicados en el notebook anterior; se incluyen acá para que el módulo sea autocontenido. Las reglas son la versión corregida: negación de acceso anticipado, `a partir de`, cualitativas de reseñas y concurrencia, `exclusivamente`, y sin el falso patrón `car\w*` que capturaba «características».

In [ ]:
COLUMNAS_CANDIDATO = ["game_id", "title", "genre", "release_year", "price_usd"]

class TituloAmbiguo(Exception):
    def __init__(self, titulo, candidatos):
        self.titulo = titulo
        self.candidatos = candidatos
        super().__init__(
            f"'{titulo}' coincide con {len(candidatos)} videojuegos. "
            "Elegí un game_id o agregá género/año/precio_max."
        )

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()
    return re.sub(r"\s+", " ", texto)

def resolver_titulo(titulo, game_id=None, genero=None, anio=None, precio_max=None):
    tit_norm = normalizar(titulo)
    candidatos = df[df["title"].map(normalizar) == tit_norm].copy()
    if game_id is not None:
        candidatos = candidatos[candidatos["game_id"] == game_id]
    if genero is not None:
        candidatos = candidatos[candidatos["genre"] == genero]
    if anio is not None:
        candidatos = candidatos[candidatos["release_year"] == anio]
    if precio_max is not None:
        candidatos = candidatos[candidatos["price_usd"] <= precio_max]
    if candidatos.empty:
        raise ValueError(f"No encontré el título '{titulo}' con esos criterios.")
    if len(candidatos) > 1:
        raise TituloAmbiguo(titulo, candidatos[COLUMNAS_CANDIDATO])
    return candidatos.iloc[0].to_dict()

@dataclass
class Intencion:
    consulta: str
    generos: list = field(default_factory=list)
    genero_semantico: list = field(default_factory=list)
    atributos_semanticos: list = field(default_factory=list)
    obligatorios: dict = field(default_factory=dict)
    preferencias: dict = field(default_factory=dict)
    titulo: str | None = None
    advertencias: list = field(default_factory=list)

    def mostrar(self):
        print("Consulta:", self.consulta)
        print("  géneros exactos   :", self.generos)
        print("  géneros semánticos:", self.genero_semantico)
        print("  atributos semánticos:", self.atributos_semanticos)
        print("  obligatorios      :", self.obligatorios)
        print("  preferencias      :", self.preferencias)
        print("  título detectado  :", self.titulo)
        for a in self.advertencias:
            print("  !", a)

PREF_MARK = (r"\b(prefiero|prefiera|idealmente|mejor si|prioriza\w*|de ser posible|"
             r"ojala|me gustaria|si es posible|preferentemente|de preferencia)\b")
OBL_MARK = (r"\b(solamente|solo|unicamente|exclusivamente|debe\w*|obligatorio|"
            r"necesito|sin|con|que tenga\w*|que sea\w*|menos de|mayor a|hasta|entre|no)\b")
OBL_MARK_CUAL = (r"\b(solamente|solo|unicamente|exclusivamente|debe\w*|"
                 r"obligatorio|necesito|siempre)\b")

def modo_local(txt, inicio, por_defecto, oblig=OBL_MARK):
    ctx = txt[max(0, inicio - 45):inicio]
    if re.search(PREF_MARK, ctx): return "preferencia"
    if re.search(oblig, ctx):     return "obligatorio"
    return por_defecto

def poner(it, var, op, valor, modo, peso=1.0):
    dest = it.obligatorios if modo == "obligatorio" else it.preferencias
    regla = {"op": op, "valor": valor, "peso": peso}
    if (var in df.columns
            and pd.api.types.is_numeric_dtype(df[var])
            and isinstance(valor, (int, float, np.integer, np.floating))):
        regla["max"] = float(df[var].max())
    dest.setdefault(var, []).append(regla)

def marcar(txt, patron):
    m = re.search(patron, txt, re.I)
    if m:
        return m, txt[:m.start()] + " " * (m.end() - m.start()) + txt[m.end():]
    return None, txt

GENERO_ALIAS = [
    ("juegos de rol", "RPG"), ("role playing", "RPG"), ("rpg", "RPG"), ("rol", "RPG"),
    ("tiro", "Shooter"), ("disparos", "Shooter"), ("accion", "Action"),
    ("aventura", "Adventure"), ("simulacion", "Simulation"),
    ("estrategia", "Strategy"), ("plataformas", "Platformer"),
    ("supervivencia", "Survival"), ("rompecabezas", "Puzzle"), ("puzzle", "Puzzle"),
    ("terror", "Horror"), ("deportes", "Sports"), ("carreras", "Racing"),
] + [(normalizar(g), g) for g in df["genre"].unique()]
GENERO_ALIAS = sorted(set(GENERO_ALIAS), key=lambda x: -len(x[0]))

BOOL_REGLAS = [
    ("has_multiplayer", 0, r"(?:sin|no)\s+(?:\w+\s+){0,2}multi\w*", "obligatorio"),
    ("has_multiplayer", 1, r"multi\w*|co-?op|jugar con amigos|en linea|online", "obligatorio"),
    ("early_access", 0, r"(?:sin|no)\s+(?:\w+\s+){0,2}acceso anticipado|(?:sin|no)\s+\w*\s*early access", "obligatorio"),
    ("early_access", 1, r"acceso anticipado|early access", "obligatorio"),
    ("is_indie", 0, r"no indie|no independiente|triple a|grandes estudios", "obligatorio"),
    ("is_indie", 1, r"indie|independientes?", "preferencia"),
    ("controller_support", 0, r"sin control|sin compatibilidad con controles|sin soporte de control", "obligatorio"),
    ("controller_support", 1, r"compatible con controles?|soporte para controles?|con control", "obligatorio"),
]

CUALITATIVAS = [
    (r"muy barat\w*|muy economic\w*|regalad\w*", "price_usd", "<=", "P10", "preferencia"),
    (r"barat\w*|economic\w*|asequible\w*", "price_usd", "<=", "P25", "preferencia"),
    (r"\bcar[oa]s?\b|\bpremium\b", "price_usd", ">=", "P75", "preferencia"),
    (r"muy bien valorad\w*|muy buenas valoraciones|excelentes valoraciones|mejor valorad\w*",
     "steam_positive_pct", ">=", "P90", "preferencia"),
    (r"bien valorad\w*|buenas valoraciones|buena valoracion|bien puntuad\w*",
     "steam_positive_pct", ">=", "P75", "preferencia"),
    (r"gran cantidad de resenas|muchas resenas|muchos reviews|gran cantidad de reviews",
     "num_reviews", ">=", "P90", "preferencia"),
    (r"bastantes resenas|bastantes reviews", "num_reviews", ">=", "P75", "preferencia"),
    (r"muy popular|muy jugad\w*|los mas jugados", "peak_concurrent_players", ">=", "P90", "preferencia"),
    (r"gran cantidad de jugadores concurrentes|muchos jugadores concurrentes|"
     r"muchos jugadores en linea|gran cantidad de jugadores",
     "peak_concurrent_players", ">=", "P90", "preferencia"),
    (r"muchos jugadores|popular|jugados?", "peak_concurrent_players", ">=", "P75", "preferencia"),
    (r"no ocupen demasiado|no ocupe demasiado|no ocupen mucho|no ocupe mucho|"
     r"que ocupen poco|que ocupe poco|poco espacio|pequen\w*|liger\w*|livian\w*|poco peso",
     "file_size_gb", "<=", "P25", "preferencia"),
]

NUM_G = r"(?P<num>\d+(?:[.,]\d+)?)"
NUM2_G = r"(?P<num2>\d+(?:[.,]\d+)?)"
UNIDAD = r"(?:usd|dolares?|gb|gigas?|anos?)"
PREFIJO1 = rf"(?P<pref>{UNIDAD})?\s*"
PREFIJO2 = rf"(?P<pref2>{UNIDAD})?\s*"
SUFIJO = rf"(?P<unidad>{UNIDAD})?"

PATRONES_NUM = [
    (rf"(?:menos de|menor a|por debajo de|debajo de|bajo)\s*(?:de\s*)?{PREFIJO1}{NUM_G}\s*{SUFIJO}", "<"),
    (rf"(?:mas de|mayor a|superior a|por encima de|encima de)\s*(?:de\s*)?{PREFIJO1}{NUM_G}\s*{SUFIJO}", ">"),
    (rf"(?:hasta|como maximo|maximo)\s*{PREFIJO1}{NUM_G}\s*{SUFIJO}", "<="),
    (rf"(?:al menos|minimo|desde|a partir de)\s*{PREFIJO1}{NUM_G}\s*{SUFIJO}", ">="),
    (rf"entre\s*{PREFIJO1}{NUM_G}\s*(?:y|a|-)\s*{PREFIJO2}{NUM2_G}\s*{SUFIJO}", "between"),
]

def variable_numerica(unidad, ctx, numeros):
    u = (unidad or "").lower()
    if u.startswith("usd") or u.startswith("dol"): return "price_usd"
    if u.startswith("gb") or u.startswith("gig"): return "file_size_gb"
    if u.startswith("ano"): return "release_year"
    if numeros and all(1900 <= n <= 2100 for n in numeros): return "release_year"
    if re.search(r"precio|usd|dolar|costar|pagar", ctx): return "price_usd"
    if re.search(r"gb|giga|tamano|ocupar|peso|espacio", ctx): return "file_size_gb"
    if re.search(r"ano|lanzamiento|estreno|reciente|antigu", ctx): return "release_year"
    if re.search(r"concurren|jugando|en linea", ctx): return "peak_concurrent_players"
    if re.search(r"resenas|reviews", ctx): return "num_reviews"
    if re.search(r"propietarios|owners", ctx): return "owners_k"
    if re.search(r"metacritic", ctx): return "metacritic_score"
    return None

def detectar_titulo(consulta):
    n = normalizar(consulta)
    coinc = [t for t in df["title"].unique() if normalizar(t) in n]
    return max(coinc, key=len) if coinc else None

### 5.2 `parsear`

Salida: géneros exactos, géneros/atributos semánticos, condiciones obligatorias y preferencias. En modo comparativo («parecido a», «del mismo estilo»), un booleano con «aunque no», «sin importar» o «abierto a» pasa a ser un atributo semántico y **no** un filtro obligatorio.

In [ ]:
def parsear(consulta):
    txt = normalizar(consulta)
    it = Intencion(consulta=consulta)
    it.titulo = detectar_titulo(consulta)

    semantic_mode = bool(re.search(
        r"\b(parecid\w*|similar\w*|semejante|tipo|del mismo estilo)\b", txt))
    sin_necesidad = bool(re.search(
        r"(?:aunque|pero)\s+no|no necesariamente|sin importar|abierto a|"
        r"no tiene que|no requiere", txt))
    resto = txt

    for alias, gen in GENERO_ALIAS:
        m, resto2 = marcar(resto, rf"\b{re.escape(alias)}\b")
        if m:
            destino = it.genero_semantico if semantic_mode else it.generos
            if gen not in destino: destino.append(gen)
            resto = resto2

    for var, valor, patron, default in BOOL_REGLAS:
        m, resto2 = marcar(resto, patron)
        if m and var not in it.obligatorios and var not in it.preferencias:
            if semantic_mode and sin_necesidad:
                it.atributos_semanticos.append((var, valor))
            else:
                poner(it, var, "=", valor, modo_local(txt, m.start(), default))
            resto = resto2

    for patron, var, op, pk, default in CUALITATIVAS:
        m, resto2 = marcar(resto, patron)
        if m and var not in it.obligatorios and var not in it.preferencias:
            poner(it, var, op, P[var][pk],
                  modo_local(txt, m.start(), default, oblig=OBL_MARK_CUAL))
            resto = resto2

    for patron, op in PATRONES_NUM:
        for m in list(re.finditer(patron, resto, re.I)):
            gd = m.groupdict()
            nums = [float(gd[k].replace(",", ".")) for k in ("num", "num2") if gd.get(k)]
            unidad = gd.get("pref") or gd.get("pref2") or gd.get("unidad") or ""
            ctx = txt[max(0, m.start() - 40):m.end() + 20]
            var = variable_numerica(unidad, ctx, nums)
            if var is None:
                it.advertencias.append(f"No pude asignar un atributo a '{m.group(0)}'.")
                continue
            valor = (nums[0], nums[1]) if op == "between" and len(nums) > 1 else nums[0]
            poner(it, var, op, valor, modo_local(txt, m.start(), "obligatorio"))
            resto = resto[:m.start()] + " " * (m.end() - m.start()) + resto[m.end():]

    if "steam_positive_pct" in it.preferencias or "steam_positive_pct" in it.obligatorios:
        it.preferencias.setdefault("num_reviews", []).append(
            {"op": ">=", "valor": P["num_reviews"]["P25"], "peso": 0.5})
        it.advertencias.append("Valoración contextualizada con num_reviews >= P25.")
    if "peak_concurrent_players" in it.preferencias:
        it.advertencias.append(
            "'Jugadores' interpretado como concurrentes; aclarar si significa propietarios.")
    return it

## 6. Consultas de prueba y verificación del parser

Las 20 consultas definitivas. La salida compacta permite confirmar de un vistazo los casos corregidos (Q03, Q07, Q10, Q13, Q14, Q18, Q19, Q20).

In [ ]:
def resumen(it):
    f = lambda d: {k: [r["op"] + str(r["valor"]) for r in v] for k, v in d.items()}
    return (f"gen={it.generos} genSem={it.genero_semantico} "
            f"attrSem={it.atributos_semanticos} obl={f(it.obligatorios)} "
            f"pref={f(it.preferencias)} tit={it.titulo}")

pruebas = [
    "Busco exclusivamente juegos RPG.",
    "Quiero juegos que tengan multijugador.",
    "Juegos que no tengan acceso anticipado.",
    "Busco juegos que cuesten menos de 15 dólares.",
    "Juegos de menos de 10 GB que tengan soporte para controles.",
    "Busco juegos baratos, preferentemente independientes.",
    "Quiero juegos con muy buenas valoraciones en Steam.",
    "Prefiero juegos con muchos jugadores concurrentes.",
    "Busco juegos pequeños, que ocupen poco espacio, y con buenas valoraciones.",
    "Quiero juegos económicos, pero priorizando aquellos con muchas reseñas.",
    "Busco juegos parecidos a los RPG, pero estoy abierto a otros géneros.",
    "Quiero juegos similares a Iron Signal.",
    "Busco videojuegos independientes con características similares a los juegos de estrategia.",
    "Quiero juegos parecidos a los multijugador, aunque no necesariamente tengan ese modo.",
    "Busco juegos del mismo estilo que Iron Signal, pero que sean más baratos.",
    "Busco RPG multijugador por menos de 20 dólares.",
    "Quiero juegos independientes, baratos y con buenas valoraciones.",
    "Busco juegos populares que no ocupen demasiado espacio.",
    "Quiero juegos lanzados a partir de 2022, preferentemente económicos y con soporte para controles.",
    "Busco juegos con muchas reseñas y una gran cantidad de jugadores concurrentes, sin importar el género.",
]

for i, q in enumerate(pruebas, 1):
    print(f"Q{i:02d}", resumen(parsear(q)))

## 7. Filtros, preferencias y ranking

`satisfaccion` usa una **meseta**: todo valor que cumple el umbral puntúa 1,0. En precio, por debajo del P10 se baja a 0,90 para evitar que los juegos gratuitos copen el ranking. La señal estructural excluye las variables ya usadas por la consulta para no contarlas dos veces.

In [ ]:
def comparar(serie, op, valor):
    if op == "=":       return serie == valor
    if op == "<":       return serie < valor
    if op == "<=":      return serie <= valor
    if op == ">":       return serie > valor
    if op == ">=":      return serie >= valor
    if op == "between": return serie.between(valor[0], valor[1])
    raise ValueError(f"Operador no soportado: {op}")

def aplicar_filtros(df_base, it):
    """Condiciones obligatorias: AND entre variables, OR dentro de géneros."""
    mask = pd.Series(True, index=df_base.index)
    if it.generos:
        mask &= df_base["genre"].isin(it.generos)
    for var, reglas in it.obligatorios.items():
        for r in reglas:
            mask &= comparar(df_base[var], r["op"], r["valor"])
    return df_base[mask].copy(), mask

TOPES = {c: float(df[c].max()) for c in df.select_dtypes(include=[np.number]).columns}
MISSING_MODE = "neutral"

def satisfaccion(x, regla, var=None):
    op, v = regla["op"], regla["valor"]
    if pd.isna(x):
        return 0.5 if MISSING_MODE == "neutral" else 0.0
    if op == "=":
        return float(x == v)
    if op == "between":
        lo, hi = v
        if lo <= x <= hi: return 1.0
        ancho = hi - lo
        if ancho <= 0: return 0.0
        dist = lo - x if x < lo else x - hi
        return float(max(0.0, 1 - dist / ancho))
    if not isinstance(v, (int, float)):
        return 0.5

    tope_ref = regla.get("max") or TOPES.get(var)
    tope = min(2 * v, tope_ref) if tope_ref else 2 * v

    if op in ("<", "<="):
        if var == "price_usd" and x < P["price_usd"]["P10"]:
            return 0.90        # por debajo del P10: atípico, sin recompensa plena
        if x <= v: return 1.0  # meseta P10..umbral
        if tope <= v: return 0.0
        return float(np.clip(1 - (x - v) / (tope - v), 0, 1))
    if op in (">", ">="):
        if x >= v: return 1.0  # meseta arriba del umbral
        if v <= 0: return 1.0
        return float(np.clip(x / v, 0, 1))
    return 0.5

def score_preferencias(fila, it):
    vals, pesos = [], []
    for var, reglas in it.preferencias.items():
        for r in reglas:
            vals.append(satisfaccion(fila[var], r, var))
            pesos.append(r["peso"])
    return float(np.average(vals, weights=pesos)) if vals else 0.5

def score_estructural(cand, it):
    """Calidad/popularidad para desempatar similitudes idénticas."""
    usadas = set(it.preferencias) | set(it.obligatorios)
    pop = [v for v in ["owners_k", "num_reviews", "peak_concurrent_players"]
           if v not in usadas]
    cal = [v for v in ["steam_positive_pct", "metacritic_score"]
           if v not in usadas]

    def media_rank(cols):
        if not cols:
            return pd.Series(0.5, index=cand.index)
        return pd.concat([cand[c].rank(pct=True) for c in cols], axis=1).mean(axis=1)

    return 0.5 * media_rank(pop) + 0.5 * media_rank(cal)

### 7.1 `buscar`: ranking híbrido con explicaciones

`S = (w_sem·sim + w_pref·pref + w_base·base) / Σw`. Si hay título de referencia, `w_sem = 2`; si hay preferencias, `w_pref = min(1 + 0,5·n, 3)`. Los títulos ambiguos devuelven un listado de candidatas para elegir con `game_id_referencia`.

In [ ]:
COLS_OUT = ["game_id", "title", "genre", "release_year", "price_usd",
            "file_size_gb", "steam_positive_pct", "num_reviews",
            "metacritic_score", "owners_k", "peak_concurrent_players",
            "has_multiplayer", "is_indie", "early_access",
            "score", "sim", "score_pref", "score_base", "por_que"]

def explicar(fila, it):
    """Explicación basada solo en reglas aplicadas y puntajes calculados."""
    partes = []
    if it.generos:
        partes.append("género " + "/".join(it.generos))
    for var, reglas in it.obligatorios.items():
        for r in reglas:
            partes.append(f"{var} {r['op']} {r['valor']}")
    for var, reglas in it.preferencias.items():
        for r in reglas:
            partes.append(
                f"{var} {r['op']} {r['valor']} (cumple {satisfaccion(fila[var], r, var):.2f})"
            )
    return " · ".join(partes)

def buscar(consulta, n=10, game_id_referencia=None, w_base=0.5):
    it = parsear(consulta)

    cand, _ = aplicar_filtros(base, it)
    if cand.empty:
        print("Sin resultados: ningún juego cumple las condiciones obligatorias.")
        return None

    if it.titulo:
        if game_id_referencia is None:
            try:
                game_id_referencia = resolver_titulo(it.titulo)["game_id"]
            except TituloAmbiguo as e:
                candidatas = e.candidatos
                if it.preferencias or it.obligatorios:
                    candidatas, _ = aplicar_filtros(candidatas, it)
                print(e, "\n")
                print("Candidatas ordenadas por precio:")
                print(candidatas[COLUMNAS_CANDIDATO].sort_values("price_usd").to_string(index=False))
                print("\n→ Elegí con game_id_referencia='...' ")
                return candidatas
        if game_id_referencia not in id_a_pos:
            raise ValueError(f"game_id desconocido: {game_id_referencia}")
        q = embeddings[id_a_pos[game_id_referencia]]
    else:
        q = modelo.encode(
            "query: " + consulta,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

    pos = cand.index.to_numpy()   # base conserva el orden de los embeddings
    sims = embeddings[pos] @ q
    s_pref = cand.apply(lambda f: score_preferencias(f, it), axis=1).to_numpy()
    s_base = score_estructural(cand, it).to_numpy()

    n_pref = sum(len(v) for v in it.preferencias.values())
    w_sem = 2.0 if it.titulo else 1.0
    w_pref = min(1.0 + 0.5 * n_pref, 3.0) if n_pref else 0.0
    denominador = w_sem + w_pref + w_base

    score = (w_sem * sims + w_pref * s_pref + w_base * s_base) / denominador

    out = cand.copy()
    out["sim"] = sims
    out["score_pref"] = s_pref
    out["score_base"] = s_base
    out["score"] = score

    if game_id_referencia is not None:
        out = out[out["game_id"] != game_id_referencia]

    out = out.sort_values("score", ascending=False).head(n)
    out["por_que"] = out.apply(lambda f: explicar(f, it), axis=1)

    print(f"Intención: {len(cand)} candidatos | pesos sem={w_sem}, "
          f"pref={w_pref:.1f}, base={w_base}")
    return out[COLS_OUT]

### 7.2 Verificación final de casos corregidos

Q06 y Q17 deben mostrar precios mayoritariamente en 2,99–4,99 (banda P10–P25), sin concentración en 0,00.

In [ ]:
for i in [6, 17]:
    r = buscar(pruebas[i - 1], n=10)
    print(f"\nQ{i:02d} precios:", sorted(r["price_usd"].tolist()))
    print(r[["game_id", "title", "price_usd", "score_pref", "score_base"]].head(6).to_string(index=False))

## 8. Benchmark definitivo

20 consultas con relevancia 0/1/2 y gold completo (sin muestreo). Q11/Q13/Q14 usan gold con alternativas semánticas; Q12/Q15 usan `G00001` como referencia explícita por la ambigüedad de «Iron Signal».

In [ ]:
def mascara(conds, df=None):
    df = base if df is None else df
    m = pd.Series(True, index=df.index)
    for col, val in conds.items():
        if isinstance(val, tuple):
            op, x = val
            m &= comparar(df[col], op, x)
        elif isinstance(val, list):
            m &= df[col].isin(val)          # OR dentro de una columna
        else:
            m &= df[col] == val
    return m

def gold_desde_mascaras(m2, m1, excluir=None):
    if excluir is not None:
        m2 = m2 & (base["game_id"] != excluir)
        m1 = m1 & (base["game_id"] != excluir)
    g2 = base.loc[m2, "game_id"].tolist()
    g1 = base.loc[m1 & ~m2, "game_id"].tolist()
    return {**{g: 2 for g in g2}, **{g: 1 for g in g1}}

def construir_gold(conds2, conds1, n2=None, n1=None, semilla=42):
    m2 = mascara(conds2)
    m1 = mascara(conds1) & ~m2
    g2 = base.loc[m2, "game_id"].to_numpy()
    g1 = base.loc[m1, "game_id"].to_numpy()

    def elegir(arr, n):
        if n is None or len(arr) <= n:
            return arr
        return np.random.default_rng(semilla).choice(arr, size=n, replace=False)

    return {**{g: 2 for g in elegir(g2, n2)},
            **{g: 1 for g in elegir(g1, n1)}}

p10 = lambda v: P[v]["P10"]
p25 = lambda v: P[v]["P25"]
p50 = lambda v: P[v]["P50"]
p75 = lambda v: P[v]["P75"]
p90 = lambda v: P[v]["P90"]

ESPEC = [
 dict(q=pruebas[0],  g2={"genre":"RPG","metacritic_score":(">=",p90("metacritic_score"))},
                     g1={"genre":"RPG","metacritic_score":(">=",p75("metacritic_score"))}),
 dict(q=pruebas[1],  g2={"has_multiplayer":1,"steam_positive_pct":(">=",p90("steam_positive_pct"))},
                     g1={"has_multiplayer":1,"steam_positive_pct":(">=",p75("steam_positive_pct"))}),
 dict(q=pruebas[2],  g2={"early_access":0,"metacritic_score":(">=",p90("metacritic_score"))},
                     g1={"early_access":0,"metacritic_score":(">=",p75("metacritic_score"))}),
 dict(q=pruebas[3],  g2={"price_usd":("<=",p10("price_usd"))},
                     g1={"price_usd":("<",15)}),
 dict(q=pruebas[4],  g2={"file_size_gb":("<=",p25("file_size_gb")),"controller_support":1,"steam_positive_pct":(">=",p75("steam_positive_pct"))},
                     g1={"file_size_gb":("<",10),"controller_support":1}),
 dict(q=pruebas[5],  g2={"is_indie":1,"price_usd":("<=",p25("price_usd"))},
                     g1={"is_indie":1,"price_usd":("<=",p50("price_usd"))}),
 dict(q=pruebas[6],  g2={"steam_positive_pct":(">=",p90("steam_positive_pct"))},
                     g1={"steam_positive_pct":(">=",p75("steam_positive_pct"))}),
 dict(q=pruebas[7],  g2={"peak_concurrent_players":(">=",p90("peak_concurrent_players"))},
                     g1={"peak_concurrent_players":(">=",p75("peak_concurrent_players"))}),
 dict(q=pruebas[8],  g2={"file_size_gb":("<=",p25("file_size_gb")),"steam_positive_pct":(">=",p75("steam_positive_pct"))},
                     g1={"file_size_gb":("<=",p50("file_size_gb")),"steam_positive_pct":(">=",p75("steam_positive_pct"))}),
 dict(q=pruebas[9],  g2={"price_usd":("<=",p25("price_usd")),"num_reviews":(">=",p90("num_reviews"))},
                     g1={"price_usd":("<=",p25("price_usd")),"num_reviews":(">=",p75("num_reviews"))}),

 # Q11: parecido a RPG, abierto a otros géneros
 dict(q=pruebas[10], g2={"genre":"RPG","steam_positive_pct":(">=",p75("steam_positive_pct"))},
                     g1={"genre":["Strategy","Adventure"],"steam_positive_pct":(">=",p75("steam_positive_pct"))}),

 # Q12: referencia ambigua → game_id explícito
 dict(q=pruebas[11], ref="G00001",
      gold=lambda: gold_desde_mascaras(
          base["title"] == "Iron Signal",
          (base["genre"] == "RPG") & (base["is_indie"] == 0) &
          (base["has_multiplayer"] == 0) & (base["controller_support"] == 1),
          excluir="G00001")),

 # Q13: indie + similar a Strategy
 dict(q=pruebas[12], g2={"genre":"Strategy","is_indie":1},
                     g1={"genre":["Simulation","Puzzle","RPG"],"is_indie":1,
                         "steam_positive_pct":(">=",p75("steam_positive_pct"))}),

 # Q14: similar a multijugador sin exigirlo
 dict(q=pruebas[13], gold=lambda: gold_desde_mascaras(
          (base["has_multiplayer"] == 1) &
          (base["steam_positive_pct"] >= p75("steam_positive_pct")),
          (base["has_multiplayer"] == 0) &
          (base["steam_positive_pct"] >= p75("steam_positive_pct")) &
          (base["peak_concurrent_players"] >= p75("peak_concurrent_players")))),

 # Q15: mismo estilo pero más barato
 dict(q=pruebas[14], ref="G00001",
      gold=lambda: gold_desde_mascaras(
          (base["title"] == "Iron Signal") &
          (base["price_usd"] <= p25("price_usd")),
          (base["title"] == "Iron Signal") & (base["price_usd"] <= 10),
          excluir="G00001")),

 dict(q=pruebas[15], g2={"genre":"RPG","has_multiplayer":1,"price_usd":("<=",p25("price_usd"))},
                     g1={"genre":"RPG","has_multiplayer":1,"price_usd":("<",20)}),
 dict(q=pruebas[16], g2={"is_indie":1,"price_usd":("<=",p25("price_usd")),"steam_positive_pct":(">=",p75("steam_positive_pct"))},
                     g1={"is_indie":1,"price_usd":("<=",p25("price_usd"))}),
 dict(q=pruebas[17], g2={"peak_concurrent_players":(">=",p90("peak_concurrent_players")),"file_size_gb":("<=",p25("file_size_gb"))},
                     g1={"peak_concurrent_players":(">=",p75("peak_concurrent_players")),"file_size_gb":("<=",p50("file_size_gb"))}),
 dict(q=pruebas[18], g2={"release_year":(">=",2022),"price_usd":("<=",p25("price_usd")),"controller_support":1},
                     g1={"release_year":(">=",2022),"controller_support":1}),
 dict(q=pruebas[19], g2={"num_reviews":(">=",p90("num_reviews")),"peak_concurrent_players":(">=",p90("peak_concurrent_players"))},
                     g1={"num_reviews":(">=",p75("num_reviews")),"peak_concurrent_players":(">=",p75("peak_concurrent_players"))}),
]

BENCH = [
    {"q": e["q"], "gold": e["gold"]() if "gold" in e else construir_gold(e["g2"], e["g1"]),
     "ref": e.get("ref")}
    for e in ESPEC
]

print("Consultas:", len(BENCH))
print("Tamaños de gold:", [len(b["gold"]) for b in BENCH])

## 9. Evaluación

Cuatro estrategias: **estructurada** (solo preferencias), **semántica** (similitud pura), **híbrida** (pesos dinámicos) y **estática** (pesos fijos como control).

In [ ]:
def dcg(rel):
    return sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rel))

def evaluar(resultados, gold, k):
    ids = resultados["game_id"].tolist()[:k]
    rel = [gold.get(g, 0) for g in ids]
    ideal = sorted(gold.values(), reverse=True)[:k]
    denom = dcg(ideal)
    return {
        f"P@{k}": sum(r > 0 for r in rel) / k,
        f"R@{k}": sum(r > 0 for r in rel) / len(gold) if gold else 0.0,
        f"NDCG@{k}": dcg(rel) / denom if denom > 0 else 0.0,
    }

def metricas(resultados, gold):
    out = {}
    for k in (5, 10):
        out.update(evaluar(resultados, gold, k))
    return out

def ranking(estrategia, consulta, ref=None, n=10, w_base=0.5):
    it = parsear(consulta)

    if estrategia == "semantica":
        cand = base.copy()
    else:
        cand, _ = aplicar_filtros(base, it)
    if cand.empty:
        return base.iloc[0:0].copy()

    if ref or it.titulo:
        gid = ref or resolver_titulo(it.titulo)["game_id"]
        q = embeddings[id_a_pos[gid]]
    else:
        q = modelo.encode("query: " + consulta,
                          normalize_embeddings=True, convert_to_numpy=True)

    pos = cand.index.to_numpy()
    sims = embeddings[pos] @ q
    n_pref = sum(len(v) for v in it.preferencias.values())

    if n_pref:
        s_pref = cand.apply(lambda f: score_preferencias(f, it), axis=1).to_numpy()
    else:
        s_pref = np.full(len(cand), 0.5)
    s_base = score_estructural(cand, it).to_numpy()

    if estrategia == "estructurada":
        w_sem, w_pref = 0.0, (2.0 if n_pref else 0.0)
    elif estrategia == "semantica":
        w_sem, w_pref, w_base = 1.0, 0.0, 0.001   # desempate casi nulo
    elif estrategia == "hibrida":
        w_sem = 2.0 if (ref or it.titulo) else 1.0
        w_pref = min(1.0 + 0.5 * n_pref, 3.0) if n_pref else 0.0
    elif estrategia == "estatica":
        w_sem, w_pref = 1.0, (1.0 if n_pref else 0.0)
    else:
        raise ValueError(estrategia)

    den = w_sem + w_pref + w_base
    score = (w_sem * sims + w_pref * s_pref + w_base * s_base) / den

    out = cand.copy()
    out["sim"], out["score_pref"], out["score_base"], out["score"] = (
        sims, s_pref, s_base, score
    )
    if ref:
        out = out[out["game_id"] != ref]
    out = out.sort_values("score", ascending=False).head(n)
    out.attrs["candidatos"] = len(cand)
    return out

ESTRATEGIAS = ["estructurada", "semantica", "hibrida", "estatica"]

filas = []
for i, b in enumerate(BENCH, start=1):
    for strat in ESTRATEGIAS:
        t0 = time.perf_counter()
        res = ranking(strat, b["q"], ref=b["ref"], n=10)
        elapsed = time.perf_counter() - t0
        fila = metricas(res, b["gold"])
        fila.update({
            "consulta": i,
            "estrategia": strat,
            "candidatos": len(base) if res.empty else res.attrs["candidatos"],
            "tiempo_s": round(elapsed, 4),
        })
        filas.append(fila)

RES = pd.DataFrame(filas)
COLUMNAS = ["P@5", "P@10", "R@5", "R@10", "NDCG@5", "NDCG@10",
            "candidatos", "tiempo_s"]
RES.groupby("estrategia")[COLUMNAS].mean().round(3)

In [ ]:
pivot = RES.pivot(index="consulta", columns="estrategia", values="NDCG@10")
display(pivot.round(3))

# Contar todos los empates, no solo el primero con idxmax
ganas_reales = pivot.eq(pivot.max(axis=1), axis=0).sum().sort_values(ascending=False)
print("=== GANADOR POR CONSULTA (incluye empates) ===")
print(ganas_reales.to_string())

delta = pd.DataFrame({
    "hibrida - estructurada": pivot["hibrida"] - pivot["estructurada"],
    "hibrida - estatica":     pivot["hibrida"] - pivot["estatica"],
    "hibrida - semantica":    pivot["hibrida"] - pivot["semantica"],
})
print("\n=== DELTAS PAREADOS (promedio) ===")
print(delta.mean().round(4).to_string())
print("\n=== CONSULTAS DONDE LA SEMÁNTICA SUPERA A LA HÍBRIDA ===")
print(delta[delta["hibrida - semantica"] < 0].round(3).to_string())

In [ ]:
def evaluar_missing(modo):
    global MISSING_MODE
    MISSING_MODE = modo
    return pd.DataFrame([
        metricas(ranking("hibrida", b["q"], ref=b["ref"], n=10), b["gold"])
        for b in BENCH
    ]).mean()

base_original = base
rng = np.random.default_rng(7)
base = base.copy()
base.loc[rng.random(len(base)) < 0.10, "steam_positive_pct"] = np.nan

res_neutral = evaluar_missing("neutral")
res_penal = evaluar_missing("penalizar")

base = base_original
MISSING_MODE = "neutral"

print("=== DATOS FALTANTES SIMULADOS (10% en steam_positive_pct) ===")
pd.concat(
    [res_neutral.rename("neutral"), res_penal.rename("penalización")], axis=1
).round(3)

## 10. Resultados finales y conclusiones

### Métricas agregadas (20 consultas)

| Estrategia | P@5 | P@10 | R@5 | R@10 | NDCG@5 | NDCG@10 | Candidatos | Tiempo |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Estructurada | 0,82 | 0,815 | 0,004 | 0,009 | 0,714 | 0,713 | 6.326 | 0,137 s |
| Estática | 0,89 | 0,870 | 0,009 | 0,013 | 0,816 | 0,792 | 6.326 | 0,134 s |
| **Híbrida** | **0,91** | **0,880** | **0,013** | **0,017** | **0,825** | **0,800** | 6.326 | 0,134 s |
| Semántica pura | 0,47 | 0,415 | 0,022 | 0,035 | 0,334 | 0,305 | 8.000 | 0,142 s |

Ganadores por consulta **contando empates**: híbrida 15, estructurada 14, estática 14, semántica 3. Deltas pareados: híbrida − estructurada **+0,087**, híbrida − estática **+0,008**, híbrida − semántica **+0,494**.

### Hallazgos

1. **La estrategia híbrida es la mejor** (NDCG@10 0,800). La semántica pura es la peor en promedio (0,305), pero es perfecta en Q12, Q13 y Q15: consultas con juego de referencia o similitud de género.
2. **Los filtros estructurados fallan en comparaciones**: Q11 (0,037), Q12 (0,000), Q13 (0,050), Q15 (0,000). Ahí el embedding aporta la señal que falta.
3. **Los pesos dinámicos aportan +0,008**, concentrados en Q15 (+0,089) y Q12 (+0,028), justamente las consultas con título de referencia.
4. **Colapso semántico**: 8.000 documentos pero solo 7.302 únicos con título y **192 sin título** (12 géneros × 2⁴). El embedding no desempata dentro de esos grupos; lo hace `score_base`.
5. **Dominio del título**: los «similares a Iron Signal» son otros `Iron Signal` con similitud 0,98–0,999. Los booleanos apenas influyen semánticamente y deben resolverse con filtros.
6. **Precio por banda**: la recompensa lineal concentraba resultados en USD 0,00. Se adoptó el rango P10–P25 (2,99–4,99), con 0,90 por debajo del P10. Las métricas no cambiaron; cambió la calidad perceptible de Q06/Q17.
7. **Nulos**: con 10 % simulado en `steam_positive_pct`, tratarlos como neutros o penalizarlos da métricas idénticas (NDCG@10 = 0,808).

### Limitaciones

- Gold construido por reglas con umbrales más estrictos que el parser; 14 de 20 consultas llegan a NDCG 1,0 (benchmark parcialmente saturado).
- Sin descripciones narrativas: se comparan metadatos, no experiencias de juego.
- Títulos ambiguos requieren `game_id` explícito o desambiguación por atributos.
- `owners_k`, `num_reviews` y `peak_concurrent_players` se agrupan en una dimensión de popularidad sin medir su correlación.

### Conclusión

La arquitectura híbrida está justificada: los filtros resuelven las condiciones obligatorias, el embedding aporta en consultas de similitud y referencia, y el ranking dinámico decide cuándo darle más peso. El módulo queda completo con el benchmark definitivo y las correcciones de interpretación aplicadas.